In [15]:
import pandas as pd
import numpy as np
import warnings
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning

In [ ]:
df = pd.read_csv("")
ts = df["Close"].apply(np.log).diff().dropna()

In [ ]:
def integration_order(series, d_max):
    d = 0
    while True:
        p_val = adfuller(series, autolag="AIC")[1]

        if p_val <= 0.05:
            return d
        
        series = series.diff()

        d += 1

        if d > d_max:
            return d

In [ ]:
def train(ts):
    ts = ts.dropna()

    p_range = range(0, 6)
    q_range = range(0, 6)

    aicc_matrix = np.full((len(p_range), len(q_range)), np.inf)

    best_model = None
    best_order = None
    best_aicc = np.inf

    for i, p in enumerate(p_range):
        for j, q in enumerate(q_range):
            try:
                with warnings.catch_warnings():
                    warnings.filterwarnings("ignore", category=ConvergenceWarning)
                    warnings.filterwarnings("ignore", message="Non-stationary starting autoregressive parameters")
                    warnings.filterwarnings("ignore", message="Non-invertible starting MA parameters")
                    
                    model = ARIMA(ts, order=(p, 0, q)).fit()

                converged = model.mle_retvals.get("converged", False)
                if not converged:
                    continue

                k = len(model.params)
                n = len(ts)

                if n - k - 1 <= 0:
                    continue

                aicc = model.aic + (2 * k**2 + 2 * k) / (n - k - 1)
                aicc_matrix[i, j] = aicc

                if aicc < best_aicc:
                    best_aicc = aicc
                    best_order = (p, 0, q)
                    best_model = model

            except Exception:
                continue

    print("Best order:", best_order)
    print("Best AICc:", best_aicc)

    return best_model, best_order, best_aicc, aicc_matrix





1      -0.004219
2      -0.027869
3      -0.018651
4       0.091982
5       0.010050
          ...   
1354   -0.034912
1355   -0.013568
1356    0.040685
1357    0.012957
1358    0.009153
Name: Close, Length: 1358, dtype: float64
